# nanoAgent — the capstone, end to end

Companion to [Post 5b: nanoAgent](../posts/05b-nanoagent.qmd).

This notebook runs the reference implementation that wires the whole
curriculum together: a Bayes-consistent controller that holds one calibrated
belief and, at each step, takes the action whose expected gain beats its cost.
Routing, adaptive compute, and graceful stopping all emerge from that rule.

**You'll do (~20 minutes):**
1. Run nanoAgent and watch it beat every fixed strategy.
2. Trace its decisions on one question of each type.
3. See routing emerge from value of information.
4. Watch calibration drive control, and adaptive compute fall out for free.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.agent import SimulatedWorld, NanoAgent, run_agent

## 1. Run it

A calibrated controller on a calibrated world. Compare to the answer-only
baseline (an agent that never acts).

In [ ]:
world = SimulatedWorld(n_questions=4000, overconfidence=1.0, seed=0)
agent = NanoAgent(calibration_T=1.0)
acc, cost, traces = run_agent(world, agent, seed=1)

answer_only = NanoAgent(max_steps=0)
acc0, _, _ = run_agent(world, answer_only, seed=1)

print(f"answer-only : {acc0:.3f}")
print(f"nanoAgent   : {acc:.3f}   (mean actions/question: {np.mean([len(t.actions) for t in traces]):.2f})")

## 2. Watch it decide

Trace one question of each type. Notice the agent takes a different action --
and a different *number* of actions -- for each, all from the one rule.

In [ ]:
rng = np.random.default_rng(7)
shown = set()
for qid in range(world.n_questions):
    qt = world.types[qid]
    if qt in shown:
        continue
    shown.add(qt)
    tr = agent.solve(world, qid, rng)
    beliefs = " -> ".join(f"{b:.2f}" for b in tr.beliefs)
    print(f"[{qt:11s}] actions={tr.actions}  belief: {beliefs}  correct={tr.correct}")
    if len(shown) == 3:
        break

## 3. Routing emerges from value of information

No rule says "use the tool on calculation." The controller computes the
expected gain of each capability and the home capability simply wins.

In [ ]:
route = {t: Counter() for t in ("knowledge", "calculation", "reasoning")}
for qid, tr in enumerate(traces):
    route[world.types[qid]][tr.actions[0] if tr.actions else "answer"] += 1
for qt, c in route.items():
    total = sum(c.values())
    top = c.most_common(2)
    print(f"{qt:12s} -> " + ", ".join(f"{a} {n/total:.0%}" for a, n in top))

## 4. The adaptive controller beats every fixed strategy

Each capability helps only its home type, so no fixed strategy wins
everywhere. The adaptive controller routes and beats them all.

In [ ]:
def fixed_accuracy(world, strategy, seed=1):
    rng = np.random.default_rng(seed); correct = 0
    for qid in range(world.n_questions):
        ans, _ = world.generate(qid, rng)
        if strategy == "tool": ans = world.use_tool(qid, rng)
        elif strategy == "retrieve": ans = world.retrieve(qid, rng)
        elif strategy == "reflect": _, ans = world.reflect(qid, ans, rng)
        elif strategy == "vote": ans = world.vote(qid, 5, rng)
        correct += world.is_correct(qid, ans)
    return correct / world.n_questions

strats = ["answer", "tool", "retrieve", "reflect", "vote"]
vals = [fixed_accuracy(world, s) for s in strats] + [acc]
labels = strats + ["nanoAgent"]
colors = ["#888","#dd8452","#3a7ebf","#c44e52","#9467bd","#55a467"]
plt.bar(labels, vals, color=colors)
for i, v in enumerate(vals): plt.text(i, v+0.01, f"{v:.2f}", ha="center")
plt.ylabel("accuracy"); plt.ylim(0, 0.95); plt.show()

## 5. Calibration drives control

When the generator is overconfident, an agent that trusts it stops too early.
An agent that temperature-scales its belief first keeps deciding well.

In [ ]:
betas = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]
naive, calib = [], []
for b in betas:
    w = SimulatedWorld(n_questions=4000, overconfidence=b, seed=0)
    naive.append(run_agent(w, NanoAgent(calibration_T=1.0), seed=1)[0])
    calib.append(run_agent(w, NanoAgent(calibration_T=b), seed=1)[0])
plt.plot(betas, calib, "o-", color="#55a467", label="calibrated (T=β)")
plt.plot(betas, naive, "s-", color="#c44e52", label="naive (trusts raw conf)")
plt.xlabel("generator overconfidence β"); plt.ylabel("accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 6. Adaptive compute, for free

Value-of-information stopping spends more actions on hard questions and fewer
on easy ones -- with no explicit difficulty signal.

In [ ]:
w = SimulatedWorld(n_questions=6000, overconfidence=1.0, seed=0)
_, _, tr = run_agent(w, NanoAgent(calibration_T=1.0), seed=1)
nact = np.array([len(t.actions) for t in tr])
acc_q = np.array([w._gen_accuracy(q) for q in range(w.n_questions)])
order = np.argsort(acc_q)
for lo, hi, lab in [(0,0.33,"hard"),(0.33,0.66,"medium"),(0.66,1.0,"easy")]:
    idx = order[int(lo*len(order)):int(hi*len(order))]
    print(f"{lab:7s} (base acc {acc_q[idx].mean():.2f}): mean actions = {nact[idx].mean():.2f}")

## What you built

nanoAgent is the whole curriculum in one controller:

- **belief** = a calibrated posterior over "is my answer right?" (Post 4c),
- **capabilities** = tools (3b), retrieval (4a), reflection (4b), voting (3a/5a),
- **the rule** = take the action whose expected gain beats its cost — value of
  information (3b, 4c), the Bayes-consistent control of Post 5a.

Routing, adaptive compute, and graceful stopping all emerged from that one
rule, and the controller reused the earlier posts' analytic results as its
value estimates. An agent, in the end, is a controller that knows what it
believes, knows what its actions are worth, and chooses accordingly.

That completes the series. Thank you for building it from scratch.